# Register Model

## Notebook Overview

- Start Execution
- Install and Import Libraries
- Configure Settings
- Register and Log the Model to MLFlow

In [1]:
%%time

%pip install -r ../requirements.txt --quiet

Note: you may need to restart the kernel to use updated packages.
CPU times: user 17.8 ms, sys: 4.31 ms, total: 22.1 ms
Wall time: 808 ms


In [2]:
MIN_TOTAL_RAM_GB = 16
MIN_TOTAL_VRAM_GB = 8


from ai_studio_blueprint_kit.memory_guard import run_memory_check_notebook


run_memory_check_notebook(
    min_total_ram_gb=MIN_TOTAL_RAM_GB,
    min_total_vram_gb=MIN_TOTAL_VRAM_GB,
)

# Start Execution

In [3]:
import logging
import time

# Configure logger
logger: logging.Logger = logging.getLogger("run_workflow_logger")
logger.setLevel(logging.INFO)
logger.propagate = False  # Prevent duplicate logs from parent loggers

# Set formatter
formatter: logging.Formatter = logging.Formatter(
    fmt="%(asctime)s - %(levelname)s - %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S"
)

# Configure and attach stream handler
stream_handler: logging.StreamHandler = logging.StreamHandler()
stream_handler.setFormatter(formatter)
logger.addHandler(stream_handler)

In [4]:
start_time = time.time()  

logger.info("Notebook execution started.")

2026-04-16 22:28:45 - INFO - Notebook execution started.


# Install and Import Libraries

In [ ]:
import datetime
import os
import uuid
import base64
import sys
from typing import Dict, Any, List
from pathlib import Path
import pandas as pd
import warnings
import mlflow
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.llms import LlamaCpp
from langchain_core.callbacks import CallbackManager, StreamingStdOutCallbackHandler
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline, HuggingFaceEndpoint
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda, RunnableMap
from langchain_core.documents import Document
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

# Define the relative path to the 'src' directory (one level up from current working directory)
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))

from src.mlflow import Logger
from src.utils import (
    get_context_window, 
    dynamic_retriever, 
    format_docs_with_adaptive_context,
    login_huggingface,
    load_config,
    load_secrets,
    load_secrets_to_env,
    clean_code
)
from src.prompt_templates import format_rag_chatbot_prompt

# Configure Settings

In [6]:
# ------------------------ Suppress Verbose Logs ------------------------
warnings.filterwarnings("ignore")

In [7]:
# Configuration parameters for the RAG chatbot
MODEL_SOURCE = "local"  # Can be: "local", "hugging-face-local", "hugging-face-cloud"
CHATBOT_SERVICE_NAME = "VanillaRAGChatbot"

# Configuration paths
CONFIG_PATH = "../configs/config.yaml"
SECRETS_PATH = "../configs/secrets.yaml"
DATA_PATH = "../data"
MLFLOW_EXPERIMENT_NAME = "AIStudio-Chatbot-Experiment"
MLFLOW_RUN_NAME = "AIStudio-Chatbot-Run"
DEMO_FOLDER = "../demo"
MLFLOW_MODEL_NAME = "AIStudio-Chatbot-Model"

In [8]:
# Load secrets from secrets.yaml file (if it exists) into environment
if Path(SECRETS_PATH).exists():
    load_secrets_to_env(SECRETS_PATH)
else:
    print(f"No secrets file found at {SECRETS_PATH}; relying on preexisting environment")

# Retrieve secrets from environment
try:
    secrets = load_secrets()
except ValueError:
    secrets = {}

# Load configuration and secrets
config = load_config(CONFIG_PATH)

print("✅ Configuration loaded successfully")
print("✅ Secrets loaded successfully")

No secrets file found at ../configs/secrets.yaml; relying on preexisting environment
✅ Configuration loaded successfully
✅ Secrets loaded successfully


# Register and Log the Model to MLFlow 

In this section, we demonstrate how to deploy a RAG-based chatbot service with integrated Galileo Protect and Observe capabilities. This service provides a REST API endpoint that allows users to query the knowledge base with natural language questions, upload new documents to the knowledge base, and manage conversation history, all with built-in safeguards against sensitive information and toxicity. This service encapsulates all the functionality we developed in this notebook, including the document retrieval system and the RAG-based question answering capabilities. It demonstrates how to use our Logger from the src/mlflow directory. 

In [9]:
%%time

mlflow.set_tracking_uri(os.getenv("MLFLOW_TRACKING_URI", '/phoenix/mlflow'))
# === Set MLflow experiment context ===
mlflow.set_experiment(experiment_name=MLFLOW_EXPERIMENT_NAME)

# === Get model path from config ===
model_path = config.get("model_path")
if model_path and os.path.exists(model_path):
    logger.info(f"✅ Model file found at: {model_path}")
else:
    logger.info(f"⚠️ Warning: Model file not found at {model_path}. Please verify the path in config.yaml.")

logger.info(f'Starting the experiment: {MLFLOW_EXPERIMENT_NAME}')
logger.info(f"Using MLflow tracking URI: {mlflow.get_tracking_uri()}")

# === Define model input/output schema for this specific blueprint ===
from mlflow.models.signature import ModelSignature
from mlflow.types.schema import Schema, ColSpec

input_schema = Schema([
    ColSpec("string", "query"),
    ColSpec("string", "prompt"),
    ColSpec("string", "document")
])
output_schema = Schema([
    ColSpec("string", "chunks"),
    ColSpec("string", "history"), 
    ColSpec("string", "prompt"),
    ColSpec("string", "output"),
    ColSpec("boolean", "success")
])
# Create signature without param_schema for now to avoid compatibility issues
signature = ModelSignature(inputs=input_schema, outputs=output_schema)

# === Log and register model to MLflow ===
with mlflow.start_run(run_name=MLFLOW_RUN_NAME) as run:
    # Print the artifact URI for reference
    logging.info(f"Run's Artifact URI: {run.info.artifact_uri}")
    
    # Log model artifacts using custom Logger
    Logger.log_model(
        artifact_path=MLFLOW_MODEL_NAME,
        config_path=CONFIG_PATH,
        docs_path=DATA_PATH,
        secrets_dict=secrets if secrets else None,
        model_path=model_path,
        demo_folder=DEMO_FOLDER,
        signature=signature
    )

    # Construct the URI for the logged model
    model_uri = f"runs:/{run.info.run_id}/{MLFLOW_MODEL_NAME}"

    # Register the model into MLflow Model Registry
    mlflow.register_model(
        model_uri=model_uri,
        name=MLFLOW_MODEL_NAME
    )

logger.info(f"✅ Model registered successfully with run ID: {run.info.run_id}")

2026/04/16 22:28:50 INFO mlflow.tracking.fluent: Experiment with name 'AIStudio-Chatbot-Experiment' does not exist. Creating a new experiment.
2026-04-16 22:28:50 - INFO - ✅ Model file found at: /home/jovyan/datafabric/meta-llama3.1-8b-Q8/Meta-Llama-3.1-8B-Instruct-Q8_0.gguf
2026-04-16 22:28:50 - INFO - Starting the experiment: AIStudio-Chatbot-Experiment
2026-04-16 22:28:50 - INFO - Using MLflow tracking URI: /phoenix/mlflow
Successfully registered model 'AIStudio-Chatbot-Model'.
2026/04/16 22:33:20 WARNING mlflow.tracking._model_registry.fluent: Run with id 1264b11f161d4b1bbd3f3d260efc305a has no artifacts at artifact path 'AIStudio-Chatbot-Model', registering model based on models:/m-c78b922432a6464da642dea8172be61d instead
Created version '1' of model 'AIStudio-Chatbot-Model'.
2026-04-16 22:33:21 - INFO - ✅ Model registered successfully with run ID: 1264b11f161d4b1bbd3f3d260efc305a


CPU times: user 1.64 s, sys: 27.9 s, total: 29.5 s
Wall time: 4min 31s


In [10]:
end_time: float = time.time()
elapsed_time: float = end_time - start_time
elapsed_minutes: int = int(elapsed_time // 60)
elapsed_seconds: float = elapsed_time % 60

logger.info(f"⏱️ Total execution time: {elapsed_minutes}m {elapsed_seconds:.2f}s")

2026-04-16 22:33:21 - INFO - ⏱️ Total execution time: 4m 36.02s


In [11]:
print("Notebook execution completed successfully.")

Notebook execution completed successfully.


Built with ❤️ using [**HP AI Studio**](https://hp.com/ai-studio).